In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Libertinus Serif', 'Libertinus Math', 'DejaVu Serif', 'Times New Roman']

PLOTS = Path("plots")
PLOTS.mkdir(exist_ok=True)


CONFOUNDED = {
    (3041, 2), (3041, 3),
    (655,  2), (655,  3),
    (20923,2), (20923,3),
    (1237, 2), (1237, 3),
}



In [16]:
records = pd.read_csv(Path("analysis/exp2_mixed_models/recordsWoutiBauti.csv"))
gold_lable = pd.read_csv(PLOTS / "jaccard_gold_labels.csv").set_index("Patient")

In [ ]:
def plot_arm_delta_new(records, gold_labels, confounded_patients, output_path="plots/arm_delta_new.pdf"):
    from matplotlib.patches import Patch

    CONDITIONS = ["1dAVb", "rbbb", "lbbb", "sb", "af", "st", "norm"]
    DIAG_MAP = {
        '1dAVb': '1° AVB', 'rbbb': 'RBBB', 'lbbb': 'LBBB',
        'sb': 'SB', 'af': 'AF', 'st': 'ST', 'norm': 'Normal'
    }

    def get_diagnosis(pid):
        if pid not in gold_labels.index:
            return 'Unknown'
        row = gold_labels.loc[pid, [c for c in CONDITIONS if c in gold_labels.columns]]
        diags = [DIAG_MAP.get(c, c) for c in row[row == 1].index.tolist()]
        return ', '.join(diags) if diags else 'Normal'

    pivot_mean = (records.groupby(["patient", "arm"])["jaccard"]
                  .mean().unstack(fill_value=np.nan))
    pivot_std  = (records.groupby(["patient", "arm"])["jaccard"]
                  .std().unstack(fill_value=np.nan))

    for arm in [1, 2, 3]:
        if arm in pivot_mean.columns:
            pivot_mean[f"delta_{arm}"] = pivot_mean[arm] - pivot_mean[0]

    pivot_mean["xai_overall"] = pivot_mean[[1, 2, 3]].mean(axis=1) - pivot_mean[0]
    pivot_mean["baseline"]     = pivot_mean[0]
    pivot_mean["baseline_std"] = pivot_std[0]

    pivot_mean = pivot_mean.sort_values("xai_overall", ascending=False)
    patients   = pivot_mean.index.tolist()
    n_rows     = len(patients)

    pivot_mean["diagnosis"] = [get_diagnosis(p) for p in patients]

    # Heatmap for baselines
    palette_hex = ['#ee3e32', '#f68838', '#fbb021', '#b1cf85', "#6b9f53"]

    def get_discrete_baseline_color(val):
        if val <= 0.20:
            return palette_hex[0]
        elif val <= 0.40:
            return palette_hex[1]
        elif val <= 0.60:
            return palette_hex[2]
        elif val <= 0.80:
            return palette_hex[3]
        else:
            return palette_hex[4]

    # Typography & Figure Setup
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Libertinus Serif', 'Libertinus Math', 'DejaVu Serif', 'Times New Roman']

    fig, axes = plt.subplots(1, 5, figsize=(10.4, max(5.8, n_rows * 0.31)), sharey=True,
                             gridspec_kw={'width_ratios': [1.25, 1, 1, 1, 1], 'wspace': 0.16})
    fig.patch.set_facecolor('#FFFFFF')

    COLOR_POS = "#6ab25a"      
    COLOR_NEG = '#ee3e32'      
    COLOR_CONF = '#8E9AA7'     
    COLOR_STRIPE = '#F8FAFC'   
    COLOR_BORDER = '#CFD8DC'   
    COLOR_DARK_TEXT = '#1E293B'
    COLOR_ZERO_TEXT = '#94A3B8'

    # Add Zebra striping to visualise rows
    for ax in axes:
        for r in range(n_rows):
            if r % 2 == 1:
                ax.axhspan(r - 0.48, r + 0.48, color=COLOR_STRIPE, zorder=0, lw=0)

    # Baseline Axis
    ax0 = axes[0]
    ax0.set_facecolor('none')
    ax0.set_title('Baseline', fontsize=8.5, fontweight='bold', pad=7, color=COLOR_DARK_TEXT)
    ax0.set_xlim(0, 1.25)
    ax0.set_xticks([0, 0.5, 1.0])
    ax0.set_xticklabels(['0', '0.5', '1.0'], fontsize=8, color=COLOR_DARK_TEXT)
    ax0.set_xlabel('Jaccard Score', fontsize=10, color=COLOR_DARK_TEXT, labelpad=4)
    ax0.set_ylim(-0.55, n_rows - 0.45)
    ax0.invert_yaxis()

    y_labels = [f"{p}  ({pivot_mean.loc[p, 'diagnosis']})" for p in patients]
    ax0.set_yticks(range(n_rows))
    ax0.set_yticklabels(y_labels, fontsize=8, color=COLOR_DARK_TEXT)
    ax0.tick_params(left=True, bottom=True, color=COLOR_BORDER, length=3)

    for r, p in enumerate(patients):
        b = pivot_mean.loc[p, 'baseline']
        s = pivot_mean.loc[p, 'baseline_std']
        if np.isnan(b):
            continue

        bar_color = get_discrete_baseline_color(b)
        ax0.barh(r, b, color=bar_color, alpha=1.0, height=0.62, zorder=2, edgecolor='none')

        upper_err = 0
        if not np.isnan(s) and s > 0:
            lower_err = min(s, b)
            upper_err = min(s, 1.0 - b)

            ax0.errorbar(b, r, xerr=[[lower_err], [upper_err]], fmt='none',
                         color='#0F172A', capsize=2, linewidth=0.75, zorder=3)

        # Position text after the bar / error bar
        right_bound = b + upper_err + 0.03
        ax0.text(right_bound, r, f"{b:.2f}", va='center', ha='left', fontsize=8,
                 color=COLOR_DARK_TEXT, zorder=5)

    # Delta subplots
    cols = ['xai_overall', 'delta_1', 'delta_2', 'delta_3']
    titles = [
        'Overall vs. Baseline',
        'Prediction support',
        'Case-based support',
        'Combined support'
    ]

    for idx, (ax, col, title) in enumerate(zip(axes[1:], cols, titles)):
        ax.set_facecolor('none')
        ax.set_title(title, fontsize=10, fontweight='bold', pad=7, color=COLOR_DARK_TEXT)
        ax.set_xlim(-0.75, 0.88)
        ax.set_xticks([-0.5, 0, 0.5])
        ax.set_xticklabels(['-0.5', '0', '+0.5'], fontsize=8, color=COLOR_DARK_TEXT)
        ax.set_xlabel('Δ Jaccard', fontsize=10, color=COLOR_DARK_TEXT, labelpad=4)
        
        ax.tick_params(left=False, bottom=True, color=COLOR_BORDER, length=3)

        arm_num = idx if col != 'xai_overall' else None

        for r, p in enumerate(patients):
            d = pivot_mean.loc[p, col]
            if np.isnan(d):
                continue

            is_conf = (arm_num is not None) and ((p, arm_num) in confounded_patients)

            if is_conf:
                c = COLOR_CONF
                hatch = '///'
                edgec = '#475569'
            elif d >= 0:
                c = COLOR_POS
                hatch = None
                edgec = COLOR_POS
            else:
                c = COLOR_NEG
                hatch = None
                edgec = COLOR_NEG

            ax.barh(r, d, color=c, alpha=0.9, height=0.62, zorder=3,
                    hatch=hatch, edgecolor=edgec, linewidth=0.5)

            val_str = f"{d:+.2f}"
            abs_d = abs(d)

            # Position text strictly before or after the bar
            if abs_d < 0.005:
                ax.text(0.025, r, '0.00', va='center', ha='left', fontsize=8, color=COLOR_ZERO_TEXT, zorder=5)
            elif d > 0:
                ax.text(d + 0.03, r, val_str, va='center', ha='left', fontsize=8,
                        color=COLOR_DARK_TEXT, zorder=5)
            else:
                ax.text(d - 0.03, r, val_str, va='center', ha='right', fontsize=8,
                        color=COLOR_DARK_TEXT, zorder=5)
            ax.axvline(0, color=COLOR_BORDER, linewidth=0.75, zorder=4)

    for ax in axes:
        for spine in ax.spines.values():
            spine.set_color(COLOR_BORDER)
            spine.set_linewidth(0.8)

    legend_elements = [
        #Patch(facecolor='#fbb021', label='Baseline (Arm 0: Heatmap)', alpha=1.0),
        Patch(facecolor=COLOR_POS, label='Improvement (Δ > 0)', alpha=0.9),
        Patch(facecolor=COLOR_NEG, label='Decrease (Δ < 0)', alpha=0.9),
        Patch(facecolor=COLOR_CONF, hatch='///', edgecolor='#475569', label='Confounded (Ref ≠ Patient)', alpha=0.9),
    ]
    fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.6, 1.),
               ncol=4, frameon=False, fontsize=10, handlelength=1.4, handleheight=0.8)

    plt.tight_layout()
    plt.subplots_adjust(top=0.89)
    out_path = Path(output_path)
    plt.savefig(out_path, bbox_inches="tight")
    if out_path.suffix.lower() == '.pdf':
        plt.savefig(out_path.with_suffix('.png'), dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved to {output_path}")

plot_arm_delta_new(records, gold_lable, CONFOUNDED, output_path=PLOTS / "arm_delta_new.pdf")
#plot_patient_variance(records, gold_lable, output_path=PLOTS / "patient_variance.pdf")


NameError: name 'CONFOUNDED' is not defined

In [24]:
gold_lable

,Patient,1dAVb,rbbb,lbbb,sb,af,st,norm
0,655,0,0,0,0,0,0,1
1,1237,1,1,0,0,0,0,0
2,1710,1,0,1,0,0,0,0
3,3041,0,1,0,0,1,0,0
4,3953,0,1,0,0,0,0,0
5,8374,0,0,1,0,1,0,0
6,10114,0,0,0,0,0,0,1
7,10999,0,0,1,0,0,0,0
8,12627,0,1,0,0,0,0,0
9,13264,0,0,1,0,0,0,0
